# 第 10 课｜Softmax、数值稳定性与因果注意力

**本课目标：**沿用第 09 课的 NumPy，把 `Q @ K.T / sqrt(Dk)` 补成完整的**缩放点积注意力（scaled dot-product attention）**：

$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^\mathsf T}{\sqrt{d_k}} + M\right)V
$

这里的 `M` 是**因果掩码**：在不允许关注的位置加上负无穷，表示这些位置的权重必须为零。本课代码采用布尔掩码 `True=允许、False=禁止`，效果与上面的加性掩码等价。

**前置知识：**Python 函数、循环、判断、NumPy 的 `shape`、`@`、`.T`、`axis=-1`。如果还记不清矩阵形状，先复习第 09 课。

**预计用时：**2–3 小时。**硬件要求：**任意可运行 NumPy 的电脑；本课不需要 GPU。

**学习方法：**每个代码单元先预测结果再执行；练习区先自己写，参考答案放在后面。演示代码与配套 `.py` 一致，练习区不会阻止 Notebook 从上到下运行。

## 0. 环境准备

在**终端**（不是 Python 的 `>>>`）安装依赖：

```bash
python -m pip install numpy jupyterlab ipykernel
```

如果你已经有 Jupyter 和 NumPy，只需要继续使用当前环境。打开 Notebook 后使用 **Kernel → Restart Kernel and Run All Cells** 可以检查执行顺序。

In [2]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("NumPy 版本：", np.__version__)
print("环境检查通过；本课只使用 NumPy。")

NumPy 版本： 2.4.2
环境检查通过；本课只使用 NumPy。


## 1. 回顾：注意力分数来自点积

第 09 课实现过：

$
S=\frac{QK^\mathsf T}{\sqrt{D_k}}
$

先用 **3 个位置**、每个查询/键向量 **2 维**的小例子复习：

| 数组 | 形状 | 含义 |
|---|---|---|
| `Q` | `(Tq, Dk)=(3,2)` | 每个查询位置的向量 |
| `K` | `(Tk, Dk)=(3,2)` | 每个被关注位置的键向量 |
| `V` | `(Tk, Dv)=(3,2)` | 被汇总的信息向量 |
| `S=Q@K.T/sqrt(Dk)` | `(Tq,Tk)=(3,3)` | 每对位置的分数 |

这里 Q、K、V 是**为了理解运算而手动设置的数组**，不是从真实语言模型中提取的权重。

In [3]:
Q = np.array([[1, 0], [0, 1], [1, 1]], dtype=np.float64)
K = np.array([[1, 0], [0, 1], [1, 1]], dtype=np.float64)
V = np.array([[1, 0], [0, 2], [3, 1]], dtype=np.float64)

print("Q.shape:", Q.shape, "K.shape:", K.shape, "V.shape:", V.shape)
print("Q:\n", Q)
print("V:\n", V)

Q.shape: (3, 2) K.shape: (3, 2) V.shape: (3, 2)
Q:
 [[1. 0.]
 [0. 1.]
 [1. 1.]]
V:
 [[1. 0.]
 [0. 2.]
 [3. 1.]]


In [4]:
scores = (Q @ K.T) / np.sqrt(Q.shape[-1])
print("scores.shape:", scores.shape)
print("scores:\n", scores)
print("手算第 0 行第 0 列：", (1 * 1 + 0 * 0) / np.sqrt(2))
np.testing.assert_allclose(scores[0, 0], 1 / np.sqrt(2))

scores.shape: (3, 3)
scores:
 [[0.7071 0.     0.7071]
 [0.     0.7071 0.7071]
 [0.7071 0.7071 1.4142]]
手算第 0 行第 0 列： 0.7071067811865475


## 2. Softmax：把分数变成非负且总和为 1 的权重

对于一行分数 `s=[s1,s2,...,sn]`：

$
\operatorname{softmax}(s)_i=\frac{e^{s_i}}{\sum_j e^{s_j}}
$

经过 softmax 后：

- 每个权重**大于 0**（输入有限时）；
- 同一行的权重之和等于 `1`（允许微小浮点误差）；
- 分数越大的位置，通常得到越大的权重。

**先预测**：`[0,0,0]` 经 softmax 后应该是什么？为什么？

In [5]:
logits = np.array([0.0, 0.0, 0.0])
exp_values = np.exp(logits)
probabilities = exp_values / exp_values.sum()
print("softmax([0,0,0])：", probabilities)
print("总和：", probabilities.sum())

softmax([0,0,0])： [0.3333 0.3333 0.3333]
总和： 1.0


### 容易踩坑：直接 `np.exp` 可能溢出

如果某个分数是 `1000`，`np.exp(1000)` 会溢出成 `inf`；接着 `inf/inf` 会得到 `nan`。下面故意演示**错误写法**，只用于理解问题。

In [6]:
very_large = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    bad_exp = np.exp(very_large)
    bad_prob = bad_exp / bad_exp.sum()
print("直接 exp：", bad_exp)
print("错误计算得到：", bad_prob)
print("包含 NaN：", np.isnan(bad_prob).any())

直接 exp： [inf inf inf]
错误计算得到： [nan nan nan]
包含 NaN： True


### 正确做法：每一行先减去最大值

对同一行的每个元素减去相同常数 `c`，softmax 不会改变：

$
\frac{e^{s_i-c}}{\sum_j e^{s_j-c}}=\frac{e^{s_i}}{\sum_j e^{s_j}}
$

取 `c=max(s)`，最大元素就变成 `0`，其余元素不大于 `0`，可以避免 `exp(大正数)` 溢出。

下面是配套脚本中的 `stable_softmax()`。它接受一维或多维非空数组，默认沿**最后一轴**计算。

In [7]:
def stable_softmax(x, axis=-1):
    """沿指定轴计算 softmax；要求输入为非空且数值有限。"""
    x = np.asarray(x, dtype=np.float64)
    if x.ndim == 0 or x.size == 0:
        raise ValueError("softmax 输入必须是非空向量或数组")
    if not np.all(np.isfinite(x)):
        raise ValueError("softmax 输入必须是有限数值；掩码请使用 masked_softmax")

    # softmax(x) = softmax(x-c)，每行减去最大值可避免 exp(大数) 溢出。
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=axis, keepdims=True)

In [8]:
safe_prob = stable_softmax(very_large)
print("稳定计算的结果：", safe_prob)
print("概率之和：", safe_prob.sum())
print("与 [0,1,2] 结果一致：", np.allclose(safe_prob, stable_softmax([0, 1, 2])))
assert np.isfinite(safe_prob).all()

稳定计算的结果： [0.09   0.2447 0.6652]
概率之和： 0.9999999999999999
与 [0,1,2] 结果一致： True


### 二维 softmax：应该沿哪条轴？

`scores.shape = (Tq,Tk)`：每一**行**是一个 Query 看所有 Key 的分数。所以要沿 `axis=-1`（列所在的最后一轴）计算，**每一行**分别归一化。

如果错误地沿 `axis=0` 做 softmax，就会对每一列归一化，不符合这里的注意力定义。

In [9]:
plain_weights = stable_softmax(scores, axis=-1)
print("无掩码权重：\n", plain_weights)
print("各行总和：", plain_weights.sum(axis=-1))
print("各列总和（不要求为 1）：", plain_weights.sum(axis=0))
np.testing.assert_allclose(plain_weights.sum(axis=-1), np.ones(3))

无掩码权重：
 [[0.4011 0.1978 0.4011]
 [0.1978 0.4011 0.4011]
 [0.2483 0.2483 0.5035]]
各行总和： [1. 1. 1.]
各列总和（不要求为 1）： [0.8471 0.8471 1.3057]


## 3. 为什么语言模型需要因果掩码？

自回归语言模型在预测下一个 token 时，当前位置不能提前读取**未来**的 token。假设一条序列有 3 个位置（索引为 0、1、2）：

| 查询位置 | 允许查看的位置 |
|---|---|
| 0 | 0 |
| 1 | 0、1 |
| 2 | 0、1、2 |

用下三角布尔矩阵表示：

```text
[[ True, False, False],
 [ True,  True, False],
 [ True,  True,  True]]
```

**注意：**这里 `True` 表示**允许**，`False` 表示**禁止**。有的框架 API 恰好使用相反约定，所以真正写代码时必须查清楚。

In [10]:
def causal_mask(length):
    """返回形状 (T,T) 的下三角布尔矩阵，允许当前位置看自己和过去。"""
    if isinstance(length, (bool, np.bool_)) or not isinstance(length, (int, np.integer)) or length <= 0:
        raise ValueError("length 必须是正整数")
    return np.tril(np.ones((length, length), dtype=bool))

In [11]:
allowed = causal_mask(3)
print("allowed.dtype:", allowed.dtype)
print("allowed:\n", allowed)
print("允许关注的位置数量（每行）：", allowed.sum(axis=-1))
assert allowed.shape == (3, 3)

allowed.dtype: bool
allowed:
 [[ True False False]
 [ True  True False]
 [ True  True  True]]
允许关注的位置数量（每行）： [1 2 3]


### 常见错误：只把未来位置的分数设为 `0`

**分数为 0 不代表概率为 0。**例如，对 `[2,0,0]` 做 softmax，后两个位置仍然会拿到非零权重。因此，必须先屏蔽，再进行 softmax。

In [12]:
wrong_masked_scores = np.array([2.0, 0.0, 0.0])
print("错误地用 0 替代未来位置：", stable_softmax(wrong_masked_scores))
print("第 1、2 个位置的概率并不是 0。")

错误地用 0 替代未来位置： [0.787  0.1065 0.1065]
第 1、2 个位置的概率并不是 0。


## 4. 带因果掩码的稳定 softmax

正确做法：**禁止位置的分数视为负无穷**，再沿最后一轴做 softmax：

$
\operatorname{softmax}([2,-\infty,-\infty])=[1,0,0]
$

本课的 `masked_softmax(scores, allowed)` 做四件事：

1. 验证分数和布尔掩码的形状；
2. 检查每个 Query 至少允许关注一个 Key；
3. 用负无穷代替禁止位置，再逐行减最大值；
4. 把禁止位置的最终概率严格保持为 `0`。

为什么要检查“整行全部禁止”？因为 `softmax([-inf,-inf,...])` 没有合法的概率分布。

In [13]:
def masked_softmax(scores, allowed):
    """对最后一个轴做带掩码的稳定 softmax。

    scores: (..., Tq, Tk) 的有限分数矩阵；最后一轴是 key 位置。
    allowed: 可广播到 scores.shape 的布尔掩码；True=允许，False=禁止。
    要求每个查询位置至少能看到一个 key，避免整行 softmax 无定义。
    """
    scores = np.asarray(scores, dtype=np.float64)
    allowed = np.asarray(allowed)
    if scores.ndim < 2 or scores.size == 0 or not np.all(np.isfinite(scores)):
        raise ValueError("scores 必须是至少二维、非空、且值有限的数组")
    if not np.issubdtype(allowed.dtype, np.bool_):
        raise ValueError("allowed 必须是布尔掩码，True 表示允许关注")

    try:
        allowed = np.broadcast_to(allowed, scores.shape)
    except ValueError as error:
        raise ValueError("allowed 的形状无法广播到 scores") from error

    if not np.all(allowed.any(axis=-1)):
        raise ValueError("至少有一个查询位置看不到任何 key，无法做 softmax")

    # 禁止的位置视为 -inf；允许位置减去各行最大值后再指数化。
    masked_scores = np.where(allowed, scores, -np.inf)
    shifted = masked_scores - np.max(masked_scores, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)  # exp(-inf)=0
    weights = exp_values / np.sum(exp_values, axis=-1, keepdims=True)
    return np.where(allowed, weights, 0.0)

In [14]:
masked_weights = masked_softmax(scores, allowed)
print("原始分数：\n", scores)
print("允许位置：\n", allowed)
print("因果权重：\n", masked_weights)
print("每行总和：", masked_weights.sum(axis=-1))
print("所有未来位置权重都是零：", np.all(masked_weights[~allowed] == 0))
np.testing.assert_allclose(masked_weights.sum(axis=-1), np.ones(3))

原始分数：
 [[0.7071 0.     0.7071]
 [0.     0.7071 0.7071]
 [0.7071 0.7071 1.4142]]
允许位置：
 [[ True False False]
 [ True  True False]
 [ True  True  True]]
因果权重：
 [[1.     0.     0.    ]
 [0.3302 0.6698 0.    ]
 [0.2483 0.2483 0.5035]]
每行总和： [1. 1. 1.]
所有未来位置权重都是零： True


In [15]:
try:
    masked_softmax(scores, np.zeros((3, 3), dtype=bool))
except ValueError as error:
    print("预期中的错误：", error)

预期中的错误： 至少有一个查询位置看不到任何 key，无法做 softmax


## 5. 最后一步：权重乘以 Value

注意力的最后一步为：

\[
O=AV
\]

其中 `A` 是注意力权重（`Tq,Tk`），`V` 是值向量（`Tk,Dv`），因此输出 `O` 的形状是 `(Tq,Dv)`。

这可以理解成：对于每个 Query 位置，按**关注权重**对所有允许的 Value 做加权求和。**Value 不是概率**，它是被混合的信息向量。

先预测：因果注意力中第 0 个位置只允许看自身，因此第 0 个输出应当等于哪个 `V` 向量？

In [16]:
causal_output = masked_weights @ V
plain_output = plain_weights @ V
print("V：\n", V)
print("有掩码输出：\n", causal_output)
print("无掩码输出：\n", plain_output)
print("输出形状：", causal_output.shape)
np.testing.assert_allclose(causal_output[0], V[0])

V：
 [[1. 0.]
 [0. 2.]
 [3. 1.]]
有掩码输出：
 [[1.     0.    ]
 [0.3302 1.3395]
 [1.7587 1.    ]]
无掩码输出：
 [[1.6044 0.7967]
 [1.4011 1.2033]
 [1.7587 1.    ]]
输出形状： (3, 2)


### 手算一次第 1 行（索引为 1）

第 1 个位置只允许看 `V[0]` 和 `V[1]`。它们的权重大约是 `0.3302` 和 `0.6698`：

```text
0.3302 * [1, 0] + 0.6698 * [0, 2]
≈ [0.3302, 1.3395]
```

这一过程就是一次**加权求和**；你已经亲手构造了注意力的输出。

In [17]:
manual = masked_weights[1, 0] * V[0] + masked_weights[1, 1] * V[1]
print("手动加权：", manual)
print("矩阵乘法：", causal_output[1])
np.testing.assert_allclose(manual, causal_output[1])

手动加权： [0.3302 1.3395]
矩阵乘法： [0.3302 1.3395]


## 6. 封装为完整注意力函数

下面的函数和 `.py` 脚本使用同一份实现，明确做了输入检查：

- `Q`: `(Tq,Dk)`；`K`: `(Tk,Dk)`；`V`: `(Tk,Dv)`。
- 允许 `Dv` 和 `Dk` 不同；但 Q、K 的特征维必须相同，K、V 的位置数必须相同。
- 默认 `causal=True`，本课只实现 `Tq=Tk` 的**完整自注意力**。
- `causal=False` 时可以使用矩形分数矩阵；真实生成中带缓存的矩形因果掩码需要考虑 Query 在整条序列中的位置，本课暂不涉及。

函数返回 `(output, weights, scores)` 三项，方便观察每步计算。

In [18]:
def _as_nonempty_matrix(name, value):
    array = np.asarray(value, dtype=np.float64)
    if array.ndim != 2 or 0 in array.shape:
        raise ValueError(f"{name} 必须是非空二维矩阵")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{name} 不能包含 NaN 或无穷大")
    return array


def scaled_dot_product_attention(q, k, v, causal=True):
    """返回 (output, weights, scores)，供课堂观察中间结果。

    q: (Tq, Dk), k: (Tk, Dk), v: (Tk, Dv)
    scores: (Tq, Tk)；weights: (Tq, Tk)；output: (Tq, Dv)。
    causal=True 时仅讨论标准自注意力：Tq 必须等于 Tk。
    """
    q = _as_nonempty_matrix("Q", q)
    k = _as_nonempty_matrix("K", k)
    v = _as_nonempty_matrix("V", v)
    if q.shape[1] != k.shape[1]:
        raise ValueError("Q 和 K 的特征维度 Dk 必须相同")
    if k.shape[0] != v.shape[0]:
        raise ValueError("K 与 V 必须有相同的 key/value 位置数量 Tk")
    if causal and q.shape[0] != k.shape[0]:
        raise ValueError("本课的因果注意力只支持 Tq=Tk 的完整自注意力")

    scores = (q @ k.T) / np.sqrt(q.shape[1])
    if causal:
        weights = masked_softmax(scores, causal_mask(q.shape[0]))
    else:
        weights = stable_softmax(scores, axis=-1)
    output = weights @ v
    return output, weights, scores


def demo_qkv():
    """返回可手算的 Q、K、V；每一行代表序列中的一个位置。"""
    q = np.array([[1, 0], [0, 1], [1, 1]], dtype=np.float64)
    k = np.array([[1, 0], [0, 1], [1, 1]], dtype=np.float64)
    v = np.array([[1, 0], [0, 2], [3, 1]], dtype=np.float64)
    return q, k, v

In [19]:
output, weights, observed_scores = scaled_dot_product_attention(Q, K, V, causal=True)
print("scores.shape:", observed_scores.shape)
print("weights.shape:", weights.shape)
print("output.shape:", output.shape)
print("output:\n", output)
np.testing.assert_allclose(output, causal_output)

scores.shape: (3, 3)
weights.shape: (3, 3)
output.shape: (3, 2)
output:
 [[1.     0.    ]
 [0.3302 1.3395]
 [1.7587 1.    ]]


## 7. 验证因果性：修改未来内容，过去输出不能变

**这比只看掩码长什么样更重要。**我们将修改第 2 个位置（最后一个位置）的 K 和 V，检查第 0、1 个输出是否变化。

这里让 Q 保持不变，改变未来的 K/V，从而模拟“未来信息被修改”。真实网络中 Q/K/V 由相应的输入向量投影得到；本实验只针对注意力这一步的可见性。

In [20]:
K_changed = K.copy()
V_changed = V.copy()
K_changed[2] = [100, -50]
V_changed[2] = [500, 900]

out_before, _, _ = scaled_dot_product_attention(Q, K, V, causal=True)
out_after, _, _ = scaled_dot_product_attention(Q, K_changed, V_changed, causal=True)

print("原输出：\n", out_before)
print("修改未来后的输出：\n", out_after)
print("前两个位置是否保持一致：", np.allclose(out_before[:2], out_after[:2]))
print("最后一个位置是否改变：", not np.allclose(out_before[2], out_after[2]))
np.testing.assert_allclose(out_before[:2], out_after[:2])

原输出：
 [[1.     0.    ]
 [0.3302 1.3395]
 [1.7587 1.    ]]
修改未来后的输出：
 [[  1.       0.    ]
 [  0.3302   1.3395]
 [500.     900.    ]]
前两个位置是否保持一致： True
最后一个位置是否改变： True


In [21]:
plain_before, _, _ = scaled_dot_product_attention(Q, K, V, causal=False)
plain_after, _, _ = scaled_dot_product_attention(Q, K_changed, V_changed, causal=False)
print("没有因果掩码时，先前位置是否可能变化：", not np.allclose(plain_before[:2], plain_after[:2]))
assert not np.allclose(plain_before[:2], plain_after[:2])

没有因果掩码时，先前位置是否可能变化： True


## 8. 形状进阶：查询数量可以不同于键数量吗？

**没有因果约束时可以。**例如 `Q.shape=(2,2)`，`K.shape=(4,2)`，`V.shape=(4,3)`：

- 注意力分数和权重：`(2,4)`；
- 输出：`(2,3)`。

这能帮助你区分两个维度：`Tq` 决定输出多少个位置，`Dv` 决定每个输出向量有多少个数。

In [22]:
Q_rect = np.ones((2, 2))
K_rect = np.ones((4, 2))
V_rect = np.arange(12, dtype=float).reshape(4, 3)
rect_out, rect_w, rect_scores = scaled_dot_product_attention(Q_rect, K_rect, V_rect, causal=False)
print("rect_scores.shape:", rect_scores.shape)
print("rect_w.shape:", rect_w.shape)
print("rect_out.shape:", rect_out.shape)
assert rect_out.shape == (2, 3)

rect_scores.shape: (2, 4)
rect_w.shape: (2, 4)
rect_out.shape: (2, 3)


### 可选进阶：增加 Batch 轴

刚才的函数为了清晰，只接收二维 Q/K/V。现实中通常有一个 Batch 轴：

```text
Q: (B, T, Dk)
K: (B, T, Dk)
V: (B, T, Dv)
```

`K.T` 不再适用；需要用 `np.swapaxes(K, -1, -2)` 仅交换最后两个轴。`masked_softmax` 中的 `(T,T)` 掩码可以广播到 `(B,T,T)`。下面演示，不要求今天独立写出批处理函数。

In [23]:
Q_batch = np.stack([Q, 2 * Q], axis=0)
K_batch = np.stack([K, K], axis=0)
V_batch = np.stack([V, 2 * V], axis=0)
batch_scores = (Q_batch @ np.swapaxes(K_batch, -1, -2)) / np.sqrt(Q_batch.shape[-1])
batch_weights = masked_softmax(batch_scores, causal_mask(Q_batch.shape[1]))
batch_output = batch_weights @ V_batch
print("Q_batch.shape:", Q_batch.shape)
print("batch_scores.shape:", batch_scores.shape)
print("batch_weights.shape:", batch_weights.shape)
print("batch_output.shape:", batch_output.shape)
np.testing.assert_allclose(batch_output[0], output)

Q_batch.shape: (2, 3, 2)
batch_scores.shape: (2, 3, 3)
batch_weights.shape: (2, 3, 3)
batch_output.shape: (2, 3, 2)


## 9. 常见误区清单

| 误区 | 正确理解 |
|---|---|
| 把 `scores` 当概率 | `scores` 是实数分数；softmax 之后才是权重 |
| 对整个矩阵一起做 softmax | 需要对每个 Query 的 Key 轴分别归一化，即 `axis=-1` |
| 未来分数改成 `0` 就屏蔽了 | `exp(0)>0`，所以概率仍可能大于零 |
| 把 `False` 理解成允许 | 本课明确规定 `True=允许` |
| 给整行设置了禁用掩码 | 整行全禁用没有合法注意力分布，需要先处理 |
| 所有矩阵都必须是方阵 | Q/K/V 的特征数与位置数有不同约束，只有本课的完整因果自注意力设 `Tq=Tk` |
| 因果掩码能保证模型正确推理 | 掩码只控制信息可见性，不保证答案正确或具备逻辑推理能力 |

---

## 10. 独立练习（先做再看答案）

### 练习 A：不运行先预测

1. `softmax([0,0,0])` 是什么？
2. `causal_mask(4)` 的形状是什么？第 0 行允许看几个位置？
3. `Q:(4,8), K:(4,8), V:(4,16)`，输出是什么形状？
4. 如果把未来位置的分数直接设为 `0`，这些位置的概率是否必然为 `0`？

预测后再运行下面的核对单元。

In [ ]:
print("A1：", stable_softmax([0, 0, 0]))
print("A2 形状：", causal_mask(4).shape, "第 0 行允许数量：", causal_mask(4)[0].sum())
A_q = np.ones((4, 8)); A_k = np.ones((4, 8)); A_v = np.ones((4, 16))
print("A3 输出形状：", scaled_dot_product_attention(A_q, A_k, A_v)[0].shape)
print("A4 举例：", stable_softmax([2, 0, 0]))

### 练习 B：独立实现稳定 softmax（约 15 分钟）

写出 `stable_softmax_ex(x)`，要求：

- 输入为非空、全为有限数的 NumPy 数组，支持 1D 和 2D；
- 沿最后一轴计算；
- `stable_softmax_ex([1000,1001,1002])` 得到有限且总和为 1 的结果。

建议先回想：`np.max(..., axis=-1, keepdims=True)`、`np.exp()` 和 `np.sum()` 怎么组合。

In [ ]:
def stable_softmax_ex(x):
    # TODO：请将这里改成你的实现。
    raise NotImplementedError("练习 B 尚未完成")

In [ ]:
try:
    test_b = stable_softmax_ex([1000, 1001, 1002])
    np.testing.assert_allclose(test_b, stable_softmax([1000, 1001, 1002]))
    print("练习 B 通过：", test_b)
except NotImplementedError:
    print("练习 B 尚未完成。请先修改 stable_softmax_ex()，再运行本单元。")

### 练习 C：独立实现因果注意力（约 25 分钟）

实现 `causal_attention_ex(q,k,v)`，用前面已经实现的 `masked_softmax()` 和 `causal_mask()`。步骤：

1. 检查 Q/K/V 是二维，且 `Dk` 与 `Tk` 兼容。
2. 计算 `Q @ K.T / np.sqrt(Dk)`。
3. 构造因果掩码，再做 masked softmax。
4. 计算权重与 V 的乘积，返回 `(output, weights)`。

要求小例子得到 `(3,2)` 的输出，且第 0 个输出严格等于 `V[0]`。

In [ ]:
def causal_attention_ex(q, k, v):
    # TODO：请自己实现，建议先写形状检查，再写三步计算。
    raise NotImplementedError("练习 C 尚未完成")

In [ ]:
try:
    result_c, weight_c = causal_attention_ex(Q, K, V)
    np.testing.assert_allclose(result_c, causal_output)
    np.testing.assert_allclose(weight_c, masked_weights)
    print("练习 C 通过：", result_c.shape)
except NotImplementedError:
    print("练习 C 尚未完成。请先修改 causal_attention_ex()，再运行本单元。")

### 可选练习 D：因果性验证

把 `V[2]` 替换为一个很大的向量，重算注意力。证明 `output[0]` 和 `output[1]` 没变，而 `output[2]` 可以改变。尝试解释为什么需要**先掩码、再 softmax**。

---

## 11. 练习参考答案（建议做完再往下看）

练习 A：

1. `[1/3, 1/3, 1/3]`；
2. `(4,4)`，第 0 行只看 1 个位置；
3. `(4,16)`；
4. 不一定；通常会获得大于零的概率。

下面的参考函数使用 `_answer` 后缀，不会覆盖你的 `_ex` 实现。

In [24]:
def stable_softmax_answer(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim == 0 or x.size == 0 or not np.all(np.isfinite(x)):
        raise ValueError("请输入非空、数值有限的数组")
    shifted = x - np.max(x, axis=-1, keepdims=True)
    e = np.exp(shifted)
    return e / e.sum(axis=-1, keepdims=True)

np.testing.assert_allclose(stable_softmax_answer([1000,1001,1002]), stable_softmax([0,1,2]))
print("参考答案 B 检查通过。")

参考答案 B 检查通过。


In [25]:
def causal_attention_answer(q, k, v):
    q = _as_nonempty_matrix("Q", q)
    k = _as_nonempty_matrix("K", k)
    v = _as_nonempty_matrix("V", v)
    if q.shape[1] != k.shape[1] or k.shape[0] != v.shape[0] or q.shape[0] != k.shape[0]:
        raise ValueError("Q/K/V 的位置数或特征维不匹配")
    scores_local = (q @ k.T) / np.sqrt(q.shape[1])
    weights_local = masked_softmax(scores_local, causal_mask(q.shape[0]))
    return weights_local @ v, weights_local

out_answer, w_answer = causal_attention_answer(Q, K, V)
np.testing.assert_allclose(out_answer, causal_output)
np.testing.assert_allclose(w_answer, masked_weights)
print("参考答案 C 检查通过。")

参考答案 C 检查通过。


## 12. 自动验收：40 项固定检查

下面运行与 `.py` 脚本相同的固定检查，包括：

- softmax 的稳定性和归一化；
- 掩码方向、未来权重为 0、整行全禁用时拒绝计算；
- 注意力的形状、加权结果、因果性；
- 形状错误和非法输入。

**固定检查通过是入门验收，不是数学证明，也不代表你已经完成了练习 B/C。**

In [26]:
def run_checks():
    """固定案例检查：遇到错误直接失败，不输出虚假的“全部通过”。"""
    passed = 0

    def check_close(label, actual, expected):
        nonlocal passed
        np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12, err_msg=label)
        passed += 1

    def check_true(label, condition):
        nonlocal passed
        if not bool(condition):
            raise AssertionError(label)
        passed += 1

    def check_raises(label, func, error_type=ValueError):
        nonlocal passed
        try:
            func()
        except error_type:
            passed += 1
        else:
            raise AssertionError(f"{label}：应该抛出 {error_type.__name__}")

    # Softmax：概率、平移不变性、数值稳定性、非法数据。
    check_close("均匀概率", stable_softmax([0, 0, 0]), [1 / 3] * 3)
    check_close("极大数稳定性", stable_softmax([1000, 1001, 1002]), stable_softmax([0, 1, 2]))
    check_close("极小数稳定性", stable_softmax([-1000, -999, -998]), stable_softmax([0, 1, 2]))
    check_close("二维 softmax 按行归一", stable_softmax([[1, 2, 3], [3, 2, 1]]).sum(axis=-1), [1, 1])
    check_close("softmax 单元素", stable_softmax([5]), [1])
    check_true("输出必须有限", np.isfinite(stable_softmax([10000, 10001, 10002])).all())
    check_raises("拒绝 NaN", lambda: stable_softmax([0, np.nan]))
    check_raises("拒绝 inf", lambda: stable_softmax([0, np.inf]))
    check_raises("拒绝空数组", lambda: stable_softmax([]))

    # 因果掩码：下三角、长度为 1、非法长度。
    check_true("掩码形状", causal_mask(3).shape == (3, 3))
    check_true("下三角布尔掩码", np.array_equal(causal_mask(3), [[1, 0, 0], [1, 1, 0], [1, 1, 1]]))
    check_true("掩码 dtype", causal_mask(3).dtype == np.dtype(bool))
    check_true("T=1", np.array_equal(causal_mask(1), [[True]]))
    check_raises("T=0 非法", lambda: causal_mask(0))
    check_raises("布尔长度非法", lambda: causal_mask(True))

    # 带掩码的 softmax：有效概率行、严格屏蔽、拒绝整行禁止。
    q, k, v = demo_qkv()
    s = q @ k.T / np.sqrt(q.shape[-1])
    m = causal_mask(3)
    w = masked_softmax(s, m)
    check_close("masked 行和", w.sum(axis=-1), np.ones(3))
    check_true("未来概率严格为 0", np.array_equal(w[~m], np.zeros(3)))
    check_close("首行概率", w[0], [1, 0, 0])
    check_true("masked 输出 shape", w.shape == (3, 3))
    check_raises("禁止所有 key 非法", lambda: masked_softmax(s, np.zeros((3, 3), dtype=bool)))
    check_raises("不匹配的掩码形状", lambda: masked_softmax(s, np.ones((2, 2), dtype=bool)))
    check_raises("非布尔掩码非法", lambda: masked_softmax(s, np.ones((3, 3), dtype=int)))
    check_raises("不接受 -inf 作为原始 scores", lambda: masked_softmax(np.array([[0, -np.inf]]), np.ones((1, 2), dtype=bool)))

    # 注意力：维度、归一、因果、后续位置不影响先前输出。
    out, weights, scores = scaled_dot_product_attention(q, k, v, causal=True)
    check_true("scores shape", scores.shape == (3, 3))
    check_true("output shape", out.shape == (3, 2))
    check_close("完整注意力权重行和", weights.sum(axis=-1), [1, 1, 1])
    check_close("注意力首位置仅看自己", out[0], v[0])
    check_close("输出等于权重乘 V", out, weights @ v)
    check_close("scores 数值", scores, [[1 / np.sqrt(2), 0, 1 / np.sqrt(2)],
                                            [0, 1 / np.sqrt(2), 1 / np.sqrt(2)],
                                            [1 / np.sqrt(2), 1 / np.sqrt(2), np.sqrt(2)]])

    k2 = k.copy()
    v2 = v.copy()
    k2[2] = [100, -50]
    v2[2] = [500, 900]
    changed, _, _ = scaled_dot_product_attention(q, k2, v2, causal=True)
    check_close("未来改变不影响先前输出", changed[:2], out[:2])
    check_true("未来位置本身允许改变", not np.allclose(changed[2], out[2]))
    no_mask, no_mask_weights, _ = scaled_dot_product_attention(q, k, v, causal=False)
    check_true("无掩码时未来权重大于零", np.all(no_mask_weights[0, 1:] > 0))
    no_mask_changed, _, _ = scaled_dot_product_attention(q, k2, v2, causal=False)
    check_true("无掩码时先前输出会受未来变化影响", not np.allclose(no_mask_changed[:2], no_mask[:2]))

    # Q / K / V 形状和输入错误。
    q_rect = np.ones((2, 2))
    k_rect = np.ones((4, 2))
    v_rect = np.ones((4, 3))
    rect_out, rect_w, _ = scaled_dot_product_attention(q_rect, k_rect, v_rect, causal=False)
    check_true("非因果矩形 attention 输出", rect_out.shape == (2, 3))
    check_true("非因果矩形 attention 权重", rect_w.shape == (2, 4))
    check_raises("拒绝 Q/K 特征维度不一致", lambda: scaled_dot_product_attention(np.ones((3, 4)), k, v))
    check_raises("拒绝 K/V 位置数不一致", lambda: scaled_dot_product_attention(q, k, np.ones((2, 2))))
    check_raises("拒绝非方形的本课因果 attention", lambda: scaled_dot_product_attention(q_rect, k_rect, v_rect, causal=True))
    check_raises("拒绝一维 Q", lambda: scaled_dot_product_attention(np.ones(2), k, v))
    check_raises("拒绝含 NaN 的 Q", lambda: scaled_dot_product_attention(np.array([[np.nan, 0], [0, 1], [1, 1]]), k, v))

    print(f"通过固定检查：{passed} 项")
    return passed

In [ ]:
passed = run_checks()
assert passed == 40

## 13. 本课总结与自测

不看上面的代码，用自己的话回答这五个问题：

1. 为什么 softmax 要先减去最大值？为什么这样不改变结果？
2. 对 `scores.shape=(Tq,Tk)`，为什么 softmax 用 `axis=-1`？
3. 未来位置不能看时，为什么**不能简单把分数设为 0**？
4. Q、K、V 的形状分别为 `(Tq,Dk)`、`(Tk,Dk)`、`(Tk,Dv)` 时，输出是什么形状？
5. 修改未来的 K/V 后，因果注意力中的过去输出为什么不会改变？

**本课完成标准：**练习 B 和 C 不再出现 `NotImplementedError`；全部固定检查通过；你能清楚地解释分数、权重、掩码和输出的区别。

**下一课（第 11 课）：PyTorch 张量与自动求导**。我们会把 NumPy 的矩阵计算改写为 Torch tensor，认识 `requires_grad`、计算图与 `backward()`，为理解“模型如何学习参数”做准备。